In [1]:
!pip install cohere qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.1 MB/s eta 0:00:00


In [ ]:
import cohere
co = cohere.Client('API_KEY') # This is your trial API key

def generate_text(prompt, temp=0):
  response = co.chat(
    message=prompt,
    # thinking={
    #       "type": "enabled"
    #     },
    temperature=temp,
    # model="command-a-plus-05-2026",
    model="command-a-03-2025",
    preamble="You are a gelpful assistance",
    # connectors=[{"id": "web-search"}]
  )
  return response.text

In [17]:
user_input = "a smart desk lamp named the LumiTech Pro"
prompt = f"""Write a creative product description for {user_input}"""

response = generate_text(prompt, temp=0.6)
print(response)

**LumiTech Pro: Illuminate Your World, Elevate Your Productivity**  

Meet the **LumiTech Pro**, the smart desk lamp that’s not just a light—it’s your personal productivity partner, mood enhancer, and style statement all in one. Designed for the modern thinker, creator, and dreamer, this isn’t just illumination; it’s an experience.  

**Adaptive Lighting for Every Moment**  
Whether you’re diving into a late-night project, unwinding with a book, or setting the vibe for a creative brainstorm, the LumiTech Pro adjusts to your needs. With **auto-brightness sensing**, it mimics natural light to reduce eye strain, while its **customizable color temperatures** let you switch from crisp, cool whites for focus to warm, soothing ambers for relaxation.  

**Smart Connectivity, Seamless Control**  
Sync your LumiTech Pro with your smart home ecosystem and take command with your voice. Compatible with Alexa, Google Assistant, and Apple HomeKit, you can dim, brighten, or change the mood with a simp

In [ ]:
from IPython.utils import text
import cohere
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, Filter
from uuid import uuid4

# setting up environment using Cohere
co = cohere.Client('API_KEY')

# for the vector database
qdrant = QdrantClient(path = "qdrant_data")

# Size is dependent on the embedding model used
qdrant.create_collection(collection_name="user_doc", vectors_config=VectorParams(size=1536, distance=Distance.COSINE))

documents = ["The capital of France is Paris.",
             "Eiffel Tower is a heritage monument located inside the city of Paris.",
             "The Eiffel Tower is one of the most famous landmarks in Paris",
             "Python is a popular programming language for data science.",
             "Qdrant is a vector database used for semantic search.",
             "Cohere provides powerful language models for embedding and generation.",
             "Paris is beautiful place",
             "The Eiffel Tower is one of the most famous landmarks in Paris."]

# Calling Cohere's embedding model
embedding = co.embed(model="embed-v4.0", texts=documents).embeddings[0]

# Creating Structure
points = [ PointStruct( id= str(uuid4()), vector=embedding, payload={"text": documents[i]}) for i in range(len(documents)) ]

qdrant.upsert(collection_name="user_doc", points=points)

# Query of the User
user_query = "where is Eiffel tower located"

# Converting the query into embeddings
query_embedding = co.embed(model="embed-v4.0", texts=[user_query]).embeddings[0]

# Searching the user query into documents
search_result = qdrant.query_points(collection_name="user_doc", query=query_embedding, limit=2).points

# Search result
retrieved_text=""
for i in search_result:
  retrieved_text+=(i.payload["text"])

retrieved_text


'The Eiffel Tower is one of the most famous landmarks in Paris.Paris is beautiful place'

In [3]:
user_query_retrieved_text = f"Use the following context {retrieved_text} and answer the query {user_query}"

In [7]:
co.chat(model="command-a-03-2025", message=user_query_retrieved_text, max_tokens=50, temperature=0.3).text

'The Eiffel Tower is located in **Paris**, France. Paris is renowned for its beauty and is home to this iconic landmark, which is one of the most famous in the world.'